# Mode A: Baseline Pure NSGA-II

**Pure NSGA-II baseline** - No enhancements, no repair heuristics, no RL guidance.

This notebook is the foundation for comparing all other modes (B, C, D, E).

## 1. Imports

In [8]:
from __future__ import annotations
import random
import numpy as np
from pathlib import Path

from schedule_engine.notebooks.core import (
    load_data, create_random_individual, course_aware_crossover, smart_mutation,
    create_evaluator, get_constraint_breakdown, run_nsga2, EvolutionConfig, get_best_individual
)
from schedule_engine.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary

print("Imports successful")

Imports successful


## 2. Configuration

In [ ]:
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# GA Parameters - SAME AS MODE B1 for fair comparison
POP_SIZE = 50       # Same as B1
NGEN = 200          # Same as B1
CXPB = 0.9
MUTPB = 0.2

# Fitness weights: -1.0 = minimize both (equal weight)
FITNESS_WEIGHTS = (-1.0, -1.0)

# Evolution config
config = EvolutionConfig(
    pop_size=POP_SIZE,
    ngen=NGEN,
    cxpb=CXPB,
    mutpb=MUTPB,
    fitness_weights=FITNESS_WEIGHTS,
    verbose=True,
    log_interval=20,  # Show detailed breakdown every 20 gens
)

# Paths
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_a_baseline/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📊 Mode A: pop={POP_SIZE}, ngen={NGEN}, weights={FITNESS_WEIGHTS}")
print(f"   (Same params as Mode B1 for fair comparison)")

 Mode A: pop=50, ngen=100, weights=(-1.0, -1.0)


## 3. Load Data

In [10]:
# Load all data with single function call
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)

print(f" {data.summary()}")

️  Non-schedulable courses filtered out

CE604: BCE5A, BCE5B, BCE5C, BCE5D, BCE5E, BCE5F (0 credits/LTP)

ENCE 256: BCE4A, BCE4B, BCE4C, BCE4D, BCE4E, BCE4F (0 credits/LTP)

ENIE 254: BIE4A, BIE4B (0 credits/LTP)

ME706: BME7A, BME7B (0 credits/LTP)

16 enrollments skipped (Survey Camp, Industrial Attachment, etc.)

 Courses: 668, Groups: 74, Instructors: 181, Rooms: 67, Quanta: 42


## 4. Test Components

In [11]:
# Test individual creation
test_ind = create_random_individual(data)
print(f" Individual has {len(test_ind)} genes")

# Test evaluation
evaluate = create_evaluator(data)
test_fitness = evaluate(test_ind)
print(f" Test fitness: hard={test_fitness[0]}, soft={test_fitness[1]}")

 Individual has 713 genes
 Test fitness: hard=5364.0, soft=2870.0


## 5. Run NSGA-II Evolution

In [12]:
# Run evolution with DRY components
final_pop, stats = run_nsga2(
    data=data,
    config=config,
    create_individual_fn=create_random_individual,
    evaluate_fn=evaluate,
    crossover_fn=course_aware_crossover,
    mutate_fn=lambda ind: smart_mutation(ind, data),  # Closure over data
)

  Gen   0:  Hard=4508  Soft=2578
         HARD: [course_comple=   0 | instructor_ex= 680 | instructor_qu= 546 | instructor_ti= 491 | room_exclusiv= 939 | room_suitabil=   0 | room_time_ava=   0 | student_group=1852]
         SOFT: [instructor_sc=  20 | paired_cohort=   0 | session_conti=2220 | student_lunch= 300 | student_sched=  38]
  Gen  10:  Hard=2667  Soft=1542
         HARD: [course_comple=   0 | instructor_ex= 294 | instructor_qu= 178 | instructor_ti= 496 | room_exclusiv= 545 | room_suitabil=   0 | room_time_ava=   0 | student_group=1154]
         SOFT: [instructor_sc=  75 | paired_cohort=   0 | session_conti= 645 | student_lunch= 592 | student_sched= 230]
  Gen  20:  Hard=2088  Soft=1193
         HARD: [course_comple=   0 | instructor_ex= 175 | instructor_qu=  86 | instructor_ti= 522 | room_exclusiv= 412 | room_suitabil=   0 | room_time_ava=   0 | student_group= 893]
         SOFT: [instructor_sc= 119 | paired_cohort=   0 | session_conti= 240 | student_lunch= 470 | student_sche

## 6. Results & Visualization

In [13]:
# Get best solution
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

# Print summary
print_summary(final_pop, stats, breakdown)

# Plot results
plot_convergence(stats, OUTPUT_DIR / "mode_a_convergence.png", title_prefix="Mode A: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_a_breakdown.png", title="Mode A: Constraint Violations")


 EVOLUTION SUMMARY

 Best Solution:
   Hard Violations: 1516
   Soft Penalty:    648.0
   Feasible:         No

 Final Population (n=50):
   Feasible:     0/50 (0.0%)
   Min Hard:     1516
   Avg Hard:     1533.0
   Min Soft:     593.0
   Avg Soft:     613.5

️ Execution Time: 90.3s

 Best Solution Constraint Breakdown:
    course_completeness: 0
    instructor_exclusivity: 117
    instructor_qualifications: 17
    instructor_schedule_compactness: 76
    instructor_time_availability: 482
    paired_cohort_practical_alignment: 0
    room_exclusivity: 299
    room_suitability: 0
    room_time_availability: 0
    session_continuity: 90
    student_group_exclusivity: 601
    student_lunch_break: 260
    student_schedule_compactness: 222

   Total Hard: 1034, Total Soft: 1130.0

 Saved: ../output/mode_a_baseline/20260123_141940/mode_a_convergence.png
 Saved: ../output/mode_a_baseline/20260123_141940/mode_a_breakdown.png


/home/krishna/Desktop/schedule-engine/src/schedule_engine/notebooks/viz.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/home/krishna/Desktop/schedule-engine/src/schedule_engine/notebooks/viz.py:173: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Export Results

Generate all outputs: schedule JSON, calendar PDF, plots, and CSVs.

In [14]:
from schedule_engine.notebooks.export import export_full_results

# Export all results (schedule.json, calendar.pdf, plots, CSVs)
export_paths = export_full_results(
    population=final_pop,
    stats=stats,
    data=data,
    output_dir=OUTPUT_DIR,
    mode_name="mode_a_baseline",
)

print(f"\n All files saved to: {OUTPUT_DIR}")

 Saved: ../output/mode_a_baseline/20260123_141940/mode_a_baseline_schedule.json
 Saved: ../output/mode_a_baseline/20260123_141940/mode_a_baseline_stats.csv
 Saved: ../output/mode_a_baseline/20260123_141940/mode_a_baseline_summary.json

 All exports complete: ../output/mode_a_baseline/20260123_141940

 All files saved to: ../output/mode_a_baseline/20260123_141940
